# Text Preprocessing Pipeline for English and Amharic Text

This notebook implements a comprehensive text preprocessing pipeline that takes in text data (English and Amharic) and outputs cleaned tokens.


In [ ]:
# Import necessary libraries
import re
import string
from typing import List, Union
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
except:
    pass


In [ ]:
class TextPreprocessingPipeline:
    """
    A comprehensive text preprocessing pipeline for English and Amharic text.
    """
    
    def __init__(self, stopwords_file='amstopwords.txt'):
        """
        Initialize the preprocessing pipeline.
        
        Args:
            stopwords_file: Path to the file containing stopwords (English and Amharic)
        """
        self.stopwords_file = stopwords_file
        self.stopwords = self._load_stopwords()
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()
        
        # English stopwords from NLTK
        self.english_stopwords = set(nltk_stopwords.words('english'))
        
        # Combine all stopwords
        self.all_stopwords = self.stopwords.union(self.english_stopwords)
    
    def _load_stopwords(self) -> set:
        """
        Load stopwords from the amstopwords.txt file.
        
        Returns:
            set: Set of stopwords
        """
        stopwords = set()
        try:
            with open(self.stopwords_file, 'r', encoding='utf-8') as f:
                for line in f:
                    word = line.strip()
                    if word:  # Skip empty lines
                        stopwords.add(word.lower())
                        stopwords.add(word)  # Also add original case
        except FileNotFoundError:
            print(f"Warning: {self.stopwords_file} not found. Using only NLTK stopwords.")
        return stopwords
    
    def remove_urls(self, text: str) -> str:
        """
        Remove URLs from text.
        
        Args:
            text: Input text
            
        Returns:
            str: Text with URLs removed
        """
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\)]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        return re.sub(url_pattern, '', text)
    
    def remove_emails(self, text: str) -> str:
        """
        Remove email addresses from text.
        
        Args:
            text: Input text
            
        Returns:
            str: Text with emails removed
        """
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        return re.sub(email_pattern, '', text)
    
    def remove_numbers(self, text: str) -> str:
        """
        Remove numbers from text.
        
        Args:
            text: Input text
            
        Returns:
            str: Text with numbers removed
        """
        return re.sub(r'\d+', '', text)
    
    def remove_punctuation(self, text: str, keep_amharic_punct: bool = False) -> str:
        """
        Remove punctuation from text.
        
        Args:
            text: Input text
            keep_amharic_punct: If True, keep Amharic punctuation marks
            
        Returns:
            str: Text with punctuation removed
        """
        if keep_amharic_punct:
            # Remove only English punctuation
            return text.translate(str.maketrans('', '', string.punctuation))
        else:
            # Remove all punctuation including Amharic punctuation
            # Amharic punctuation: ፣ (comma), ፤ (semicolon), ፡ (colon)
            amharic_punct = '፣፤፡'
            all_punct = string.punctuation + amharic_punct
            return text.translate(str.maketrans('', '', all_punct))
    
    def to_lowercase(self, text: str) -> str:
        """
        Convert text to lowercase (mainly for English).
        
        Args:
            text: Input text
            
        Returns:
            str: Lowercased text
        """
        return text.lower()
    
    def remove_whitespace(self, text: str) -> str:
        """
        Remove extra whitespace from text.
        
        Args:
            text: Input text
            
        Returns:
            str: Text with normalized whitespace
        """
        # Replace multiple whitespaces with single space
        text = re.sub(r'\s+', ' ', text)
        # Remove leading and trailing whitespace
        return text.strip()
    
    def remove_stopwords(self, tokens: List[str]) -> List[str]:
        """
        Remove stopwords from token list.
        
        Args:
            tokens: List of tokens
            
        Returns:
            List[str]: Tokens with stopwords removed
        """
        return [token for token in tokens if token not in self.all_stopwords]
    
    def tokenize(self, text: str) -> List[str]:
        """
        Tokenize text into words.
        
        Args:
            text: Input text
            
        Returns:
            List[str]: List of tokens
        """
        # For mixed English and Amharic text, use word_tokenize for English parts
        # and split by whitespace for Amharic parts
        try:
            # Try NLTK tokenization first (works well for English)
            tokens = word_tokenize(text)
        except:
            # Fallback to simple whitespace splitting
            tokens = text.split()
        
        # Filter out empty tokens
        tokens = [token for token in tokens if token.strip()]
        return tokens
    
    def stem_tokens(self, tokens: List[str]) -> List[str]:
        """
        Apply stemming to tokens (mainly for English).
        
        Args:
            tokens: List of tokens
            
        Returns:
            List[str]: Stemmed tokens
        """
        stemmed = []
        for token in tokens:
            # Only stem if token contains English characters
            if re.search(r'[a-zA-Z]', token):
                stemmed.append(self.stemmer.stem(token))
            else:
                stemmed.append(token)  # Keep Amharic tokens as is
        return stemmed
    
    def lemmatize_tokens(self, tokens: List[str]) -> List[str]:
        """
        Apply lemmatization to tokens (mainly for English).
        
        Args:
            tokens: List of tokens
            
        Returns:
            List[str]: Lemmatized tokens
        """
        lemmatized = []
        for token in tokens:
            # Only lemmatize if token contains English characters
            if re.search(r'[a-zA-Z]', token):
                lemmatized.append(self.lemmatizer.lemmatize(token))
            else:
                lemmatized.append(token)  # Keep Amharic tokens as is
        return lemmatized
    
    def remove_short_tokens(self, tokens: List[str], min_length: int = 2) -> List[str]:
        """
        Remove tokens that are too short.
        
        Args:
            tokens: List of tokens
            min_length: Minimum token length
            
        Returns:
            List[str]: Tokens with short ones removed
        """
        return [token for token in tokens if len(token) >= min_length]
    
    def preprocess(self, 
                   text: str, 
                   remove_urls_flag: bool = True,
                   remove_emails_flag: bool = True,
                   remove_numbers_flag: bool = True,
                   remove_punctuation_flag: bool = True,
                   lowercase_flag: bool = True,
                   remove_stopwords_flag: bool = True,
                   stem_flag: bool = False,
                   lemmatize_flag: bool = False,
                   remove_short_tokens_flag: bool = True,
                   min_token_length: int = 2) -> List[str]:
        """
        Main preprocessing pipeline.
        
        Args:
            text: Input text (English and/or Amharic)
            remove_urls_flag: Whether to remove URLs
            remove_emails_flag: Whether to remove email addresses
            remove_numbers_flag: Whether to remove numbers
            remove_punctuation_flag: Whether to remove punctuation
            lowercase_flag: Whether to convert to lowercase
            remove_stopwords_flag: Whether to remove stopwords
            stem_flag: Whether to apply stemming
            lemmatize_flag: Whether to apply lemmatization
            remove_short_tokens_flag: Whether to remove short tokens
            min_token_length: Minimum length for tokens
            
        Returns:
            List[str]: List of preprocessed tokens
        """
        # Step 1: Remove URLs
        if remove_urls_flag:
            text = self.remove_urls(text)
        
        # Step 2: Remove emails
        if remove_emails_flag:
            text = self.remove_emails(text)
        
        # Step 3: Remove numbers
        if remove_numbers_flag:
            text = self.remove_numbers(text)
        
        # Step 4: Remove punctuation
        if remove_punctuation_flag:
            text = self.remove_punctuation(text)
        
        # Step 5: Convert to lowercase (mainly affects English)
        if lowercase_flag:
            text = self.to_lowercase(text)
        
        # Step 6: Remove extra whitespace
        text = self.remove_whitespace(text)
        
        # Step 7: Tokenize
        tokens = self.tokenize(text)
        
        # Step 8: Remove stopwords
        if remove_stopwords_flag:
            tokens = self.remove_stopwords(tokens)
        
        # Step 9: Apply stemming (if requested)
        if stem_flag and not lemmatize_flag:  # Don't do both
            tokens = self.stem_tokens(tokens)
        
        # Step 10: Apply lemmatization (if requested)
        if lemmatize_flag:
            tokens = self.lemmatize_tokens(tokens)
        
        # Step 11: Remove short tokens
        if remove_short_tokens_flag:
            tokens = self.remove_short_tokens(tokens, min_token_length)
        
        return tokens


## Example Usage


In [ ]:
# Initialize the pipeline
pipeline = TextPreprocessingPipeline(stopwords_file='amstopwords.txt')


In [ ]:
# Example 1: English text
english_text = """
Hello! This is a sample English text. It contains some numbers like 123 and 456.
You can visit https://example.com for more information. Contact us at info@example.com.
The quick brown fox jumps over the lazy dog. This is a test sentence.
"""

print("Original English Text:")
print(english_text)
print("\n" + "="*50 + "\n")

tokens = pipeline.preprocess(english_text)
print("Preprocessed Tokens:")
print(tokens)
print("\nNumber of tokens:", len(tokens))


In [ ]:
# Example 2: Amharic text
amharic_text = """
ኢትዮጵያ በአፍሪካ ምስራቅ የምትገኝ ሀገር ናት። ይህ ሀገር በታሪክ እና በባህል በጣም ሀብታም ናት።
የኢትዮጵያ ህዝብ በተለያየ ቋንቋዎች ይናገራሉ። አማርኛ የሀገሪቱ ዋና ቋንቋ ነው።
"""

print("Original Amharic Text:")
print(amharic_text)
print("\n" + "="*50 + "\n")

tokens = pipeline.preprocess(amharic_text, lowercase_flag=False)  # Don't lowercase Amharic
print("Preprocessed Tokens:")
print(tokens)
print("\nNumber of tokens:", len(tokens))


In [ ]:
# Example 3: Mixed English and Amharic text
mixed_text = """
Ethiopia (ኢትዮጵያ) is a beautiful country in East Africa. 
The capital city is Addis Ababa (አዲስ አበባ). 
ኢትዮጵያ has a rich history and culture. Visit https://ethiopia.travel for tourism info.
Contact: tourism@ethiopia.gov.et or call +251-11-123-4567.
"""

print("Original Mixed Text:")
print(mixed_text)
print("\n" + "="*50 + "\n")

tokens = pipeline.preprocess(mixed_text)
print("Preprocessed Tokens:")
print(tokens)
print("\nNumber of tokens:", len(tokens))


In [ ]:
# Example 4: Custom preprocessing options
sample_text = "This is a TEST sentence with 123 numbers and some@email.com and https://example.com"

print("Original Text:")
print(sample_text)
print("\n" + "="*50 + "\n")

# With lemmatization
tokens_lemmatized = pipeline.preprocess(sample_text, lemmatize_flag=True)
print("With Lemmatization:")
print(tokens_lemmatized)
print()

# With stemming
tokens_stemmed = pipeline.preprocess(sample_text, stem_flag=True)
print("With Stemming:")
print(tokens_stemmed)
print()

# Without removing stopwords
tokens_with_stopwords = pipeline.preprocess(sample_text, remove_stopwords_flag=False)
print("Without Removing Stopwords:")
print(tokens_with_stopwords)


## Pipeline Summary

The preprocessing pipeline performs the following steps:

1. **Remove URLs**: Removes web links from the text
2. **Remove Emails**: Removes email addresses
3. **Remove Numbers**: Removes numeric digits
4. **Remove Punctuation**: Removes punctuation marks (including Amharic punctuation: ፣, ፤, ፡)
5. **Lowercase**: Converts text to lowercase (mainly for English)
6. **Normalize Whitespace**: Removes extra spaces
7. **Tokenization**: Splits text into individual tokens/words
8. **Remove Stopwords**: Removes common stopwords (English and Amharic from amstopwords.txt)
9. **Stemming/Lemmat

ization**: Optional word normalization (for English)
10. **Remove Short Tokens**: Filters out very short tokens

All steps are configurable through the `preprocess()` method parameters.
